<a href="https://colab.research.google.com/github/avalee0215/COMPSYS-306_Project2/blob/ava_project1/COMPSYS_306_clee482.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive mount
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
import os, json, sys
from glob import glob
import numpy as np
from skimage.io import imread
from skimage.transform import resize
from tqdm import tqdm

# ---------- Config ----------
ARCHIVE_DIR = "/content/gdrive/MyDrive/CS306_2025/database/archive"
MYDATA_DIR  = os.path.join(ARCHIVE_DIR, "myData")
PROCESSED_DIR = os.path.join(ARCHIVE_DIR, "processed")

IMG_SIZE = (32, 32)
FORCE_REBUILD = False

NPZ_PATH  = os.path.join(PROCESSED_DIR, "arrays_flatten_32x32x3.npz")
LMAP_PATH = os.path.join(PROCESSED_DIR, "label_map.json")


def ensure_dir(p):
    if not os.path.exists(p):
        os.makedirs(p, exist_ok=True)

def to_rgb(img):
    """Ensure 3 channels (RGB). Lab images can be RGB or grayscale.
    - If grayscale (H, W) -> repeat to (H, W, 3)
    - If RGBA (H, W, 4)  -> drop alpha to (H, W, 3)
    """
    if img.ndim == 2:  # grayscale
        img = np.stack([img, img, img], axis=-1)
    elif img.ndim == 3 and img.shape[-1] == 4:  # RGBA
        img = img[..., :3]
    return img

def sorted_class_dirs(root):
    """Return class directories sorted by numeric id if possible (0..42)."""
    dirs = [d for d in glob(os.path.join(root, "*")) if os.path.isdir(d)]
    # try numeric sort
    try:
        dirs = sorted(dirs, key=lambda p: int(os.path.basename(p)))
    except ValueError:
        dirs = sorted(dirs)
    return dirs

# ---------- Core Loader ----------
def load_images_from_mydata(root_dir, img_size):
    """
    Step 1.A: Read images class-by-class
      - myData/<class_name>/*.*
      - resize to img_size (32x32), normalize to [0..1]
      - flatten to 1D for classical ML (SVM/MLP inputs)
    Returns:
      X (N, 32*32*3) float32 in [0,1]
      y (N,) int64
      idx_to_name: {idx:int -> class_name:str}
    """
    X, y = [], []
    idx_to_name = {}

    class_dirs = sorted_class_dirs(root_dir)
    if not class_dirs:
        raise FileNotFoundError(f"No class folders found under: {root_dir}")

    print(f"[Info] Found {len(class_dirs)} class folders under: {root_dir}")

    for idx, class_path in enumerate(class_dirs):
        class_name = os.path.basename(class_path)
        idx_to_name[idx] = class_name

        img_paths = []
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.ppm"):
            img_paths.extend(glob(os.path.join(class_path, ext)))

        if len(img_paths) == 0:
            print(f"[Warn] No images in class '{class_name}' — skipping.")
            continue

        for p in tqdm(img_paths, desc=f"Class {class_name}", leave=False):
            try:
                img = imread(p)
                img = to_rgb(img)
                # preserve_range=True keeps original [0..255] before scaling
                img = resize(img, img_size, preserve_range=True, anti_aliasing=True)
                img = img.astype("float32") / 255.0
                X.append(img)
                y.append(idx)
            except Exception as e:
                # Robustness: skip unreadable/broken images
                print(f"[Skip] {p} ({e})")

    if len(X) == 0:
        raise RuntimeError("No images loaded. Check dataset path/contents.")

    X = np.array(X, dtype=np.float32)   # (N, H, W, 3)
    y = np.array(y, dtype=np.int64)     # (N,)

    N = X.shape[0]
    X = X.reshape(N, -1)

    return X, y, idx_to_name

def save_label_map(idx_to_name, path_json):
    """Step 1.C: Save label map for report/confusion-matrix axes."""
    # also include reverse map for convenience
    data = {
        "idx_to_name": {str(k): v for k, v in idx_to_name.items()},
        "name_to_idx": {v: k for k, v in idx_to_name.items()},
    }
    with open(path_json, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

# ---------- Main ----------
def main():
    print("[Step 1] Load & Cache images (Lab style)")

    ensure_dir(PROCESSED_DIR)

    if os.path.exists(NPZ_PATH) and os.path.exists(LMAP_PATH) and not FORCE_REBUILD:
        print(f"[Info] Cache exists. Skipping rebuild.\n- {NPZ_PATH}\n- {LMAP_PATH}")
        data = np.load(NPZ_PATH)
        X, y = data["X"], data["y"]
        print(f"[Info] Loaded cached arrays: X={X.shape}, y={y.shape}")
        return

    # Load from myData
    X, y, idx_to_name = load_images_from_mydata(MYDATA_DIR, IMG_SIZE)

    # Save arrays (.npz compressed)
    np.savez_compressed(NPZ_PATH, X=X, y=y)
    print(f"[Saved] {NPZ_PATH} (X={X.shape}, y={y.shape})")

    # Save label map (.json)
    save_label_map(idx_to_name, LMAP_PATH)
    print(f"[Saved] {LMAP_PATH} (num_classes={len(idx_to_name)})")

    # Quick summary (similar to lab prints)
    classes, counts = np.unique(y, return_counts=True)
    print(f"[Summary] Classes present: {len(classes)}")
    print(f"[Summary] Samples per class (first 10): {counts[:10]} ...")

if __name__ == "__main__":
    # Make sure the dataset exists where expected
    if not os.path.isdir(MYDATA_DIR):
        print(f"[Error] myData not found at: {MYDATA_DIR}", file=sys.stderr)
        sys.exit(1)
    main()



[Step 1] Load & Cache images (Lab style)
[Info] Cache exists. Skipping rebuild.
- /content/gdrive/MyDrive/CS306_2025/database/archive/processed/arrays_flatten_32x32x3.npz
- /content/gdrive/MyDrive/CS306_2025/database/archive/processed/label_map.json
[Info] Loaded cached arrays: X=(73221, 3072), y=(73221,)


In [ ]:
import os, json, sys
import numpy as np
from sklearn.model_selection import train_test_split

# ---------- Config ----------
ARCHIVE_DIR   = "/content/gdrive/MyDrive/CS306_2025/database/archive"
PROCESSED_DIR = os.path.join(ARCHIVE_DIR, "processed")

STEP1_NPZ = os.path.join(PROCESSED_DIR, "arrays_flatten_32x32x3.npz")
OUT_SPLITS = os.path.join(PROCESSED_DIR, "splits_flatten_32x32x3.npz")

RANDOM_STATE = 42
TEST_SIZE = 0.20   # 20% for test
VAL_SIZE  = 0.10   # 10% of train portion

def main():
    print("2): Stratified Train/Val/Test Split")

    # Safety checks
    if not os.path.exists(STEP1_NPZ):
        print(f"[Error] Step 1 cache not found: {STEP1_NPZ}", file=sys.stderr)
        sys.exit(1)

    # Load cached arrays from Step 1
    data = np.load(STEP1_NPZ)
    X, y = data["X"], data["y"]
    print(f"[Info] Loaded: X={X.shape}, y={y.shape}")

    # First split -> Train/Test (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
        shuffle=True
    )
    print(f"[Split 1] Train={X_train.shape[0]}, Test={X_test.shape[0]}")

    # Second split -> Train/Val (stratified, from the train portion)
    # Val fraction relative to the *current* train size:
    val_size_abs = int(np.floor(VAL_SIZE * X_train.shape[0]))
    # Convert to fraction for train_test_split (must be in (0,1))
    val_frac = val_size_abs / X_train.shape[0] if X_train.shape[0] > 0 else 0.1

    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train,
        test_size=val_frac,
        stratify=y_train,
        random_state=RANDOM_STATE,
        shuffle=True
    )
    print(f"[Split 2] Train={X_train.shape[0]}, Val={X_val.shape[0]} (≈{VAL_SIZE*100:.0f}% of train)")


    def cls_counts(y_arr):
        cls, cnt = np.unique(y_arr, return_counts=True)
        return dict(zip(cls.tolist(), cnt.tolist()))
    print(f"[Check] Class counts (train) sample: {str(dict(list(cls_counts(y_train).items())[:5]))} ...")


    np.savez_compressed(
        OUT_SPLITS,
        X_train=X_train, y_train=y_train,
        X_val=X_val,     y_val=y_val,
        X_test=X_test,   y_test=y_test
    )
    print(f"[Saved] {OUT_SPLITS}")

if __name__ == "__main__":
    main()


In [ ]:
# PCA 2D scatter of all splits
import os, numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA

ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")
PROC   = os.path.join(ARCHIVE_DIR, "processed")
FIGS   = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(FIGS, exist_ok=True)

d = np.load(os.path.join(PROC, "splits_flatten_32x32x3.npz"))
X_all = np.vstack([d["X_train"], d["X_val"], d["X_test"]])
y_all = np.hstack([d["y_train"], d["y_val"], d["y_test"]])


rng = np.random.RandomState(42)
if X_all.shape[0] > 8000:
    idx = rng.choice(X_all.shape[0], size=8000, replace=False)
    X_sub, y_sub = X_all[idx], y_all[idx]
else:
    X_sub, y_sub = X_all, y_all

Z = PCA(n_components=2, random_state=42).fit_transform(X_sub)

plt.figure(figsize=(6.5, 5.5))
plt.scatter(Z[:,0], Z[:,1], c=y_sub, s=4, alpha=0.6, cmap="tab20")
plt.title("Traffic Signs — PCA (2D projection)")
plt.tight_layout()
plt.savefig(os.path.join(FIGS, "pca2_scatter.png"), dpi=200)
plt.close()
print("[Saved] pca2_scatter.png")


SVM

In [ ]:
import os, sys, pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---------- Config ----------
ARCHIVE_DIR    = "/content/gdrive/MyDrive/CS306_2025/database/archive"
PROCESSED_DIR  = os.path.join(ARCHIVE_DIR, "processed")
MODELS_DIR     = os.path.join(ARCHIVE_DIR, "models")

SPLITS_PATH    = os.path.join(PROCESSED_DIR, "splits_flatten_32x32x3.npz")
SCALER_PATH    = os.path.join(PROCESSED_DIR, "svm_scaler.pkl")
STD_CACHE_PATH = os.path.join(PROCESSED_DIR, "svm_arrays_standardized.npz")   # optional
FINAL_MODEL    = os.path.join(MODELS_DIR, "svm_final.pkl")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE       = 42
SUBSET_TRAIN_FRACTION = 0.25

# Baseline parameters
SVM_SETTINGS = [
    {"name": "linear", "svc": SVC(kernel="linear", C=1, random_state=RANDOM_STATE)},
    {"name": "poly",   "svc": SVC(kernel="poly",   C=1, gamma="auto", degree=3, random_state=RANDOM_STATE)},
    {"name": "rbf",    "svc": SVC(kernel="rbf",    C=1, gamma="scale", random_state=RANDOM_STATE)},
]


def standardize_train_val_test(X_train, X_val, X_test):
    """
    Step 3.A: Standardize features
      - Fit StandardScaler on TRAIN only (to avoid leakage)
      - Transform train/val/test with the same scaler
    """
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_val_std   = scaler.transform(X_val)
    X_test_std  = scaler.transform(X_test)
    return scaler, X_train_std, X_val_std, X_test_std

def stratified_small_subset(X_tr, y_tr, fraction=0.25, seed=42):
    """
    Step 3.B: Build a small stratified subset from TRAIN for fast model selection.
      - Preserve class proportions (StratifiedShuffleSplit)
      - Return X_sub, y_sub (subset of TRAIN)
    """
    if not (0 < fraction <= 1.0):
        fraction = 0.25
    sss = StratifiedShuffleSplit(n_splits=1, train_size=fraction, random_state=seed)
    idx_sub, _ = next(sss.split(X_tr, y_tr))
    return X_tr[idx_sub], y_tr[idx_sub]

def train_eval_once(model_name, svc, X_tr, y_tr, X_va, y_va, X_te, y_te, stage="SEL"):
    """
    Step 3.C: Train + evaluate one SVM
      - stage="SEL" for selection (subset train)
      - stage="FINAL" for final (full train)
    """
    print(f"\n[{stage}] Train SVM ({model_name})")
    svc.fit(X_tr, y_tr)

    # Evaluate on VAL
    yv_pred = svc.predict(X_va)
    val_acc = accuracy_score(y_va, yv_pred)
    print(f"[{stage}] Val  {model_name:>6s} | acc={val_acc:.4f}")

    # Evaluate on TEST
    yt_pred = svc.predict(X_te)
    test_acc = accuracy_score(y_te, yt_pred)
    print(f"[{stage}] Test {model_name:>6s} | acc={test_acc:.4f}")


    print(f"[{stage}] classification report (VAL):")
    print(classification_report(y_va, yv_pred, digits=4))
    print(f"[{stage}] classification report (TEST):")
    print(classification_report(y_te, yt_pred, digits=4))
    cm = confusion_matrix(y_te, yt_pred)
    print(f"[{stage}] confusion_matrix (TEST) shape={cm.shape}")

    return {"name": model_name, "val_acc": val_acc, "test_acc": test_acc, "model": svc}

# ---------- Main ----------
def main():
    print("3): SVM with small stratified subset selection ")
    print(f"[Paths] in={SPLITS_PATH}")

    # Safety
    if not os.path.exists(SPLITS_PATH):
        print(f"[Error] Step 2 splits not found: {SPLITS_PATH}", file=sys.stderr)
        sys.exit(1)

    # Load splits
    d = np.load(SPLITS_PATH)
    X_train, y_train = d["X_train"], d["y_train"]
    X_val,   y_val   = d["X_val"],   d["y_val"]
    X_test,  y_test  = d["X_test"],  d["y_test"]
    print(f"[Info] Loaded splits: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

    # Standardize to fit
    scaler, X_train_std, X_val_std, X_test_std = standardize_train_val_test(X_train, X_val, X_test)

    # Save scaler for reproducibility or inference
    with open(SCALER_PATH, "wb") as f:
        pickle.dump(scaler, f)
    print(f"[Saved] {SCALER_PATH}")

    # cache standardized arrays
    np.savez_compressed(
        STD_CACHE_PATH,
        X_train_std=X_train_std, y_train=y_train,
        X_val_std=X_val_std,     y_val=y_val,
        X_test_std=X_test_std,   y_test=y_test
    )
    print(f"[Saved] {STD_CACHE_PATH}")

    # Build a small stratified subset from TRAIN for fast model selection
    X_tr_sel, y_tr_sel = stratified_small_subset(
        X_train_std, y_train, fraction=SUBSET_TRAIN_FRACTION, seed=RANDOM_STATE
    )
    print(f"[SEL] Using subset of TRAIN: {X_tr_sel.shape[0]} samples (~{int(SUBSET_TRAIN_FRACTION*100)}%)")

    # -------- Find the best kernel --------
    selection_results = []
    for cfg in SVM_SETTINGS:
        res = train_eval_once(
            model_name=cfg["name"],
            svc=cfg["svc"],
            X_tr=X_tr_sel, y_tr=y_tr_sel,
            X_va=X_val_std, y_va=y_val,
            X_te=X_test_std, y_te=y_test,
            stage="SEL"
        )
        selection_results.append(res)

    # Pick the best by validation accuracy
    selection_results.sort(key=lambda r: (r["val_acc"], r["test_acc"]), reverse=True)
    best_name = selection_results[0]["name"]
    print("\n[SEL] Summary (subset-based):")
    for r in selection_results:
        print(f"  - {r['name']:>6s}: val_acc={r['val_acc']:.4f}, test_acc={r['test_acc']:.4f}")
    print(f"[SEL] Selected best kernel: {best_name}")

    # -------- Retrain the selected kernel on FULL TRAIN --------
    # Re-instantiate the selected SVC with the same settings
    best_cfg = next(cfg for cfg in SVM_SETTINGS if cfg["name"] == best_name)
    final_svc = SVC(**{k: v for k, v in best_cfg["svc"].get_params().items()})  # clone params

    final_res = train_eval_once(
        model_name=best_name,
        svc=final_svc,
        X_tr=X_train_std, y_tr=y_train,
        X_va=X_val_std,   y_va=y_val,
        X_te=X_test_std,  y_te=y_test,
        stage="FINAL"
    )

    # Save final trained model
    with open(FINAL_MODEL, "wb") as f:
        pickle.dump(final_svc, f)
    print(f"[Saved] {FINAL_MODEL}")

    # Final summary
    print("\n[Summary] FINAL model (retrained on full TRAIN)")
    print(f"  - kernel={best_name} | val_acc={final_res['val_acc']:.4f} | test_acc={final_res['test_acc']:.4f}")
    print("[Done] Step 3 complete.")

if __name__ == "__main__":
    main()



In [ ]:
# ===========================================
# Fast RBF-SVM
# ===========================================
import os, time, pickle, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

ARCHIVE_DIR = "/content/gdrive/MyDrive/CS306_2025/database/archive"
PROC   = os.path.join(ARCHIVE_DIR, "processed")
MODELS = os.path.join(ARCHIVE_DIR, "models")
FIGS   = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(MODELS, exist_ok=True); os.makedirs(FIGS, exist_ok=True)

# --- Load splits ---
d = np.load(os.path.join(PROC, "splits_flatten_32x32x3.npz"))
Xtr, ytr = d["X_train"], d["y_train"]
Xva, yva = d["X_val"],   d["y_val"]
Xte, yte = d["X_test"],  d["y_test"]

# --- subset for coarse search (e.g., 20% of TRAIN, stratified) ---
SEL_FRACTION = 0.20
sss = StratifiedShuffleSplit(n_splits=1, train_size=SEL_FRACTION, random_state=42)
idx_sel, _ = next(sss.split(Xtr, ytr))
Xs, ys = Xtr[idx_sel], ytr[idx_sel]
print(f"[Subset] {Xs.shape[0]} samples (~{int(SEL_FRACTION*100)}% of TRAIN)")

# --- pipeline ---
def make_pipe(C=None, gamma=None):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="rbf", cache_size=2000, C=C if C else 1.0,
                    gamma=gamma if gamma else "scale", random_state=42))
    ])

# --- coarse grid on SUBSET ---
C_list = [1, 2, 5, 10, 20, 50]
G_list = [5e-4, 1e-3, 2e-3, 5e-3]
records = []
t0 = time.time()
for i, C in enumerate(C_list):
    for j, g in enumerate(G_list):
        t1 = time.time()
        clf = make_pipe(C, g).fit(Xs, ys)
        f1 = f1_score(yva, clf.predict(Xva), average="macro")
        records.append((C, g, f1))
        print(f"[{i+1}/{len(C_list)} x {j+1}/{len(G_list)}] C={C} gamma={g} -> VAL mF1={f1:.4f} ({time.time()-t1:.1f}s)")

print(f"[Coarse search] {len(records)} fits in {time.time()-t0:.1f}s")

# pick top-k to try on FULL TRAIN
records.sort(key=lambda z: z[2], reverse=True)
topk = records[:3]
print("[Top-k from subset]:", topk)

# --- full TRAIN retrain among top-k and select final ---
best = None
for C, g, _ in topk:
    t1 = time.time()
    clf = make_pipe(C, g).fit(Xtr, ytr)
    f1 = f1_score(yva, clf.predict(Xva), average="macro")
    if (best is None) or (f1 > best[0]):
        best = (f1, C, g, clf)
    print(f"[Full train] C={C} gamma={g} -> VAL mF1={f1:.4f} ({time.time()-t1:.1f}s)")

best_f1, best_C, best_g, best_model = best
print(f"[Final] VAL macro-F1={best_f1:.4f} with C={best_C}, gamma={best_g}")

# --- TEST & save ---
yt = best_model.predict(Xte)
rep = classification_report(yte, yt, digits=3, output_dict=True)
pd.DataFrame(rep).T.to_csv(os.path.join(FIGS, "svm_rbf_full_noPCA_report_test.csv"))

cm = confusion_matrix(yte, yt)
plt.figure(figsize=(6,5)); plt.imshow(cm, aspect="auto")
plt.title("SVM RBF (no PCA) — Confusion Matrix (TEST)")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout()
plt.savefig(os.path.join(FIGS, "svm_rbf_full_noPCA_cm_test.png"), dpi=150); plt.close()

with open(os.path.join(MODELS, "svm_rbf_full_noPCA.pkl"), "wb") as f:
    pickle.dump(best_model, f)

summary = pd.DataFrame([{
    "name": "svm_rbf_full_noPCA",
    "val_macroF1": float(best_f1),
    "test_acc": float(accuracy_score(yte, yt)),
    "test_macroF1": float(f1_score(yte, yt, average="macro")),
    "best_C": best_C, "best_gamma": best_g,
}])
summary.to_csv(os.path.join(FIGS, "svm_rbf_full_noPCA_summary.csv"), index=False)
print("[Saved] model + reports + summary")


[Subset] 10543 samples (~20% of TRAIN)
[1/6 x 1/4] C=1 gamma=0.0005 -> VAL mF1=0.7424 (587.0s)
[1/6 x 2/4] C=1 gamma=0.001 -> VAL mF1=0.7503 (727.4s)
[1/6 x 3/4] C=1 gamma=0.002 -> VAL mF1=0.6890 (871.2s)
[1/6 x 4/4] C=1 gamma=0.005 -> VAL mF1=0.5228 (896.7s)
[2/6 x 1/4] C=2 gamma=0.0005 -> VAL mF1=0.8175 (560.1s)
[2/6 x 2/4] C=2 gamma=0.001 -> VAL mF1=0.8280 (672.4s)
[2/6 x 3/4] C=2 gamma=0.002 -> VAL mF1=0.7671 (818.8s)
[2/6 x 4/4] C=2 gamma=0.005 -> VAL mF1=0.6022 (880.0s)
[3/6 x 1/4] C=5 gamma=0.0005 -> VAL mF1=0.8812 (506.8s)
[3/6 x 2/4] C=5 gamma=0.001 -> VAL mF1=0.8625 (682.8s)


In [ ]:
#  SVM + PCA option comparison

import os, json, time, pickle
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")

PROC   = os.path.join(ARCHIVE_DIR, "processed")
MODELS = os.path.join(ARCHIVE_DIR, "models")
FIGS   = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(MODELS, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

# --- Load splits and label map ---
splits_p = os.path.join(PROC, "splits_flatten_32x32x3.npz")
d = np.load(splits_p)
Xtr, ytr, Xva, yva, Xte, yte = d["X_train"], d["y_train"], d["X_val"], d["y_val"], d["X_test"], d["y_test"]

lmap_p = os.path.join(PROC, "label_map.json")
with open(lmap_p, "r", encoding="utf-8") as f:
    idx2name = json.load(f)["idx_to_name"]
target_names = [idx2name[str(i)] for i in range(len(idx2name))]

# --- SVM pipeline with optional PCA ---
def make_svm(pca_dim=None, C=10, gamma="scale", random_state=42):
    """
    Build a leakage-safe pipeline:
      Train: fit(StandardScaler) -> fit(PCA?) -> fit(SVC)
      Val/Test: transform(StandardScaler) -> transform(PCA?) -> predict(SVC)
    """
    steps = [("scaler", StandardScaler())]
    if pca_dim is not None:
        steps.append(("pca", PCA(n_components=pca_dim, svd_solver="randomized", random_state=random_state)))
    steps.append(("svc", SVC(kernel="rbf", C=C, gamma=gamma, random_state=random_state)))
    return Pipeline(steps)

# --- no PCA vs PCA(100) ---
cands = [
    ("svm_rbf_c10",        make_svm(pca_dim=None, C=10, gamma="scale")),
    ("svm_rbf_c10_pca100", make_svm(pca_dim=100,  C=10, gamma="scale")),
]

rows = []
for name, clf in cands:
    print(f"\n[3C] Training {name} ...")
    t0 = time.time()
    clf.fit(Xtr, ytr)                      # Fit on TRAIN only
    train_time = time.time() - t0

    # Validation accuracy (model selection view)
    val_acc = clf.score(Xva, yva)

    # Test metrics (final reporting view)
    yhat_te = clf.predict(Xte)
    test_acc = accuracy_score(yte, yhat_te)
    rep = classification_report(yte, yhat_te, target_names=target_names, digits=3, output_dict=True)

    # Save per-model TEST report CSV (for the paper/report)
    pd.DataFrame(rep).T.to_csv(os.path.join(FIGS, f"{name}_report_test.csv"))

    # Save the trained model
    with open(os.path.join(MODELS, f"{name}.pkl"), "wb") as f:
        pickle.dump(clf, f)

    # If PCA was used, store explained variance ratio sum for documentation
    pca_ev = None
    if "pca" in dict(clf.named_steps):
        pca_ev = float(clf.named_steps["pca"].explained_variance_ratio_.sum())

    rows.append({
        "name": name,
        "val_acc": float(val_acc),
        "test_acc": float(test_acc),
        "train_time_sec": float(train_time),
        "pca": "yes" if pca_ev is not None else "no",
        "pca_explained_variance_sum": pca_ev
    })

# Save a compact CSV summary (you can drop this into a table in the report)
summary_csv = os.path.join(FIGS, "svm_pca_compare.csv")
pd.DataFrame(rows).to_csv(summary_csv, index=False)
print(f"[Saved] {summary_csv} and per-model *_report_test.csv")



[3C] Training svm_rbf_c10 ...

[3C] Training svm_rbf_c10_pca100 ...
[Saved] /content/drive/MyDrive/CS306_2025/database/archive/report/figures/svm_pca_compare.csv and per-model *_report_test.csv


Compare: linear-SVM, RBF(no pca), RBF(pca:100)

In [ ]:
# SVM FINAL REPORT — linear

import os, sys, json, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, accuracy_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split


ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")
PROC   = os.path.join(ARCHIVE_DIR, "processed")
MODELS = os.path.join(ARCHIVE_DIR, "models")
FIGS   = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(FIGS, exist_ok=True)


def load_splits(proc_dir, seed=42):
    # Try splits_* first
    for fname in ["splits_flatten_32x32x3.npz", "splits_32x32_flatten.npz", "splits.npz"]:
        p = os.path.join(proc_dir, fname)
        if not os.path.exists(p):
            continue
        d = np.load(p)
        keys = set(d.files)
        candidates = [
            ("Xtr","Xval","Xte","ytr","yval","yte"),
            ("X_train","X_val","X_test","y_train","y_val","y_test"),
            ("Xtrain","Xval","Xtest","Ytrain","Yval","Ytest"),
            ("XTr","XVal","XTe","yTr","yVal","yTe"),
        ]
        for XtrK, XvalK, XteK, ytrK, yvalK, yteK in candidates:
            if {XtrK,XvalK,XteK,ytrK,yvalK,yteK}.issubset(keys):
                return d[XtrK], d[ytrK], d[XvalK], d[yvalK], d[XteK], d[yteK]
    # Fallback: arrays_* (X,y) -> make 72/8/20
    for fname in ["arrays_flatten_32x32x3.npz", "arrays_32x32_flatten.npz", "arrays.npz"]:
        p = os.path.join(proc_dir, fname)
        if not os.path.exists(p):
            continue
        d = np.load(p)
        keys = set(d.files)
        Xkey = next((k for k in ["X","Xall","X_all","features"] if k in keys), None)
        ykey = next((k for k in ["y","yall","y_all","labels"] if k in keys), None)
        if Xkey and ykey:
            X, y = d[Xkey], d[ykey]
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)
            X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.10, stratify=y_tr, random_state=seed)
            return X_tr, y_tr, X_val, y_val, X_te, y_te
    raise FileNotFoundError("Could not load splits from processed/. Provide splits_* or arrays_* npz.")


def load_label_names(proc_dir):
    for fname in ["label_map_flat.json", "label_map.json"]:
        p = os.path.join(proc_dir, fname)
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                lm = json.load(f)
            if isinstance(lm, dict) and "idx_to_name" in lm:
                lm = lm["idx_to_name"]
            return {str(k): v for k, v in lm.items()}
    return None


def save_confmats(y_true, y_pred, labels_sorted, display_names, tag):
    for norm_tag, suffix in [(None, "raw"), ("true", "norm")]:
        cm = confusion_matrix(y_true, y_pred, labels=labels_sorted, normalize=norm_tag)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_names if display_names is not None else labels_sorted)
        fig, ax = plt.subplots(figsize=(8,6))
        disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=90)
        ax.set_title(f"{tag} — Confusion Matrix (normalize={norm_tag})")
        plt.tight_layout()
        plt.savefig(os.path.join(FIGS, f"confmat_{tag}_{suffix}.png"), dpi=170)
        plt.close(fig)

def save_perclass_f1(y_true, y_pred, class_names, tag):
    rep = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    df = pd.DataFrame(rep).T.iloc[:-3]  # drop macro/weighted/accuracy rows
    plt.figure(figsize=(max(12, len(df)*0.28), 4.5))
    df["f1-score"].plot(kind="bar")
    plt.ylim(0, 1.0); plt.ylabel("F1-score"); plt.title(f"{tag} — per-class F1 (TEST)")
    plt.tight_layout(); plt.savefig(os.path.join(FIGS, f"f1_per_class_{tag}.png"), dpi=170); plt.close()


def eval_and_export(tag, clf, Xte_raw, yte, scaler_for_linear=None, label_map=None):

    if tag.startswith("svm_linear") and scaler_for_linear is not None:
        X_in = scaler_for_linear.transform(Xte_raw)
    else:
        X_in = Xte_raw  # pipeline will scale if needed

    t0 = time.time()
    yhat = clf.predict(X_in)
    infer_time = time.time() - t0

    # Metrics
    acc = accuracy_score(yte, yhat)
    f1m = f1_score(yte, yhat, average="macro")
    pm  = precision_score(yte, yhat, average="macro", zero_division=0)
    rm  = recall_score(yte, yhat, average="macro", zero_division=0)

    # Full report CSV
    if label_map:
        class_names = [label_map.get(str(i), str(i)) for i in range(len(np.unique(yte)))]
        rep = classification_report(yte, yhat, target_names=class_names, digits=3, output_dict=True)
    else:
        rep = classification_report(yte, yhat, digits=3, output_dict=True)

    pd.DataFrame(rep).T.to_csv(os.path.join(FIGS, f"{tag}_report_test.csv"))

    with open(os.path.join(FIGS, f"{tag}_metrics.json"), "w") as f:
        json.dump({
            "model": tag,
            "test": {
                "accuracy": float(acc),
                "macro_f1": float(f1m),
                "macro_precision": float(pm),
                "macro_recall": float(rm),
                "infer_time_sec": float(infer_time)
            }
        }, f, indent=2)

    # Confusion matrices & per-class F1
    labels_sorted = np.unique(yte)
    display_names = None
    if label_map:
        display_names = [label_map.get(str(i), str(i)) for i in labels_sorted]
    save_confmats(yte, yhat, labels_sorted, display_names, tag)
    if label_map:
        class_names_full = [label_map.get(str(i), str(i)) for i in range(len(label_map))]
        save_perclass_f1(yte, yhat, class_names_full, tag)

    return {
        "model": tag,
        "test_accuracy": acc,
        "test_macro_f1": f1m,
        "test_macro_precision": pm,
        "test_macro_recall": rm,
        "infer_time_sec": infer_time
    }

# ===================== MAIN =====================
# Load data
Xtr, ytr, Xval, yval, Xte, yte = load_splits(PROC)
label_map = load_label_names(PROC)

# Load models
paths = {
    "svm_linear_final": os.path.join(MODELS, "svm_final.pkl"),
    "svm_rbf_c10":      os.path.join(MODELS, "svm_rbf_c10.pkl"),
    "svm_rbf_c10_pca100": os.path.join(MODELS, "svm_rbf_c10_pca100.pkl"),
}
models = {}
for tag, p in paths.items():
    if os.path.exists(p):
        with open(p, "rb") as f:
            models[tag] = pickle.load(f)
    else:
        print(f"[Info] skip (missing): {p}")

if "svm_linear_final" not in models:
    print("[Error] Missing linear final model: models/svm_final.pkl", file=sys.stderr)
    sys.exit(1)

# Load external scaler for linear
scaler_path = os.path.join(PROC, "svm_scaler.pkl")
scaler = None
if os.path.exists(scaler_path):
    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)
else:
    print("[Warn] svm_scaler.pkl not found. Linear SVM will be evaluated on raw inputs (accuracy may drop).")

# Evaluate all loaded SVM models on TEST
rows = []
for tag, clf in models.items():
    summary = eval_and_export(tag, clf, Xte, yte, scaler_for_linear=scaler, label_map=label_map)
    rows.append(summary)
    print(f"[Test] {tag:18s} acc={summary['test_accuracy']:.4f}  "
          f"F1={summary['test_macro_f1']:.4f}  P={summary['test_macro_precision']:.4f}  "
          f"R={summary['test_macro_recall']:.4f}  time={summary['infer_time_sec']:.2f}s")

# Save comparison CSV
comp_csv = os.path.join(FIGS, "svm_models_test_metrics.csv")
pd.DataFrame(rows).to_csv(comp_csv, index=False)
print(f"[Saved] {comp_csv}")
print("[Done] Reports, metrics, confusion matrices, and per-class F1 for SVMs.")


[Test] svm_linear_final   acc=0.9760  F1=0.9751  P=0.9745  R=0.9761  time=772.57s
[Test] svm_rbf_c10        acc=0.9849  F1=0.9846  P=0.9893  R=0.9804  time=1966.61s
[Test] svm_rbf_c10_pca100 acc=0.9763  F1=0.9777  P=0.9836  R=0.9724  time=105.84s
[Saved] /content/gdrive/MyDrive/CS306_2025/database/archive/report/figures/svm_models_test_metrics.csv
[Done] Reports, metrics, confusion matrices, and per-class F1 for SVMs.


MLP

In [ ]:


import os, sys, pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ----- Paths -----
ARCHIVE_DIR    = "/content/gdrive/MyDrive/CS306_2025/database/archive"
PROCESSED_DIR  = os.path.join(ARCHIVE_DIR, "processed")
MODELS_DIR     = os.path.join(ARCHIVE_DIR, "models")
REPORT_DIR     = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

SPLITS_PATH = os.path.join(PROCESSED_DIR, "splits_flatten_32x32x3.npz")
MLP_PATH    = os.path.join(MODELS_DIR, "mlp.pkl")

RANDOM_STATE = 42  # for reproducibility

def main():
    print("4): Train & Evaluate MLP")

    if not os.path.exists(SPLITS_PATH):
        print(f"[Error] Missing splits: {SPLITS_PATH}", file=sys.stderr)
        sys.exit(1)

    d = np.load(SPLITS_PATH)
    X_train, y_train = d["X_train"], d["y_train"]
    X_val,   y_val   = d["X_val"],   d["y_val"]
    X_test,  y_test  = d["X_test"],  d["y_test"]
    print(f"[Info] X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

    # ---- Model ----
    mlp = MLPClassifier(
        hidden_layer_sizes=(100,),  # classic lab default
        max_iter=200,               # sklearn default
        random_state=RANDOM_STATE,
        verbose=True                # show progress
        # activation='relu', solver='adam', learning_rate_init=1e-3 are defaults
    )

    print("[Train] Fitting MLP...")
    mlp.fit(X_train, y_train)

    # ---- Evaluation ----
    yv = mlp.predict(X_val)
    val_acc = accuracy_score(y_val, yv)
    print(f"[Val ] acc={val_acc*100:.2f}%")
    print(classification_report(y_val, yv, digits=4))

    yt = mlp.predict(X_test)
    test_acc = accuracy_score(y_test, yt)
    print(f"[Test] acc={test_acc*100:.2f}%")
    print(classification_report(y_test, yt, digits=4))

    # Confusion matrix numbers
    cm = confusion_matrix(y_test, yt)
    print("[Test] confusion_matrix shape:", cm.shape)

    # Save a simple loss curve if there is
    if hasattr(mlp, "loss_curve_") and len(mlp.loss_curve_) > 0:
        plt.figure()
        plt.plot(mlp.loss_curve_)
        plt.xlabel("Iteration"); plt.ylabel("Training Loss")
        plt.title("MLP Loss Curve")
        plt.tight_layout()
        out_png = os.path.join(REPORT_DIR, "mlp_loss_curve.png")
        plt.savefig(out_png, dpi=150)
        plt.close()
        print(f"[Saved] {out_png}")

    # ---- Save model ----
    with open(MLP_PATH, "wb") as f:
        pickle.dump(mlp, f)
    print(f"[Saved] {MLP_PATH}")

    print("[Summary] val_acc={:.2f}%, test_acc={:.2f}%".format(val_acc*100, test_acc*100))
    print("[Done] Step 4 complete.")

if __name__ == "__main__":
    main()


In [ ]:
# Build splits_flatten_32x32x3.npz from arrays_flatten_32x32x3.npz
import os, json, numpy as np
from sklearn.model_selection import train_test_split

ARCHIVE_DIR = "/content/gdrive/MyDrive/CS306_2025/database/archive"
PROC = os.path.join(ARCHIVE_DIR, "processed")
arrays_p = os.path.join(PROC, "arrays_flatten_32x32x3.npz")
splits_p = os.path.join(PROC, "splits_flatten_32x32x3.npz")
seed = 42

d = np.load(arrays_p)
X, y = d["X"], d["y"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)
X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.10, stratify=y_tr, random_state=seed)

np.savez_compressed(splits_p, Xtr=X_tr, ytr=y_tr, Xval=X_val, yval=y_val, Xte=X_te, yte=y_te)
print("[Saved]", splits_p)


[Saved] /content/gdrive/MyDrive/CS306_2025/database/archive/processed/splits_flatten_32x32x3.npz


MLP - comparing pipeline

In [ ]:

# MLP scaling comparison — MinMax([0,1]) vs StandardScaler(z-score)


import os, time, json, pickle, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, ConfusionMatrixDisplay)

# ----- Paths (your structure) -----
ARCHIVE_DIR = "/content/gdrive/MyDrive/CS306_2025/database/archive"
PROC   = os.path.join(ARCHIVE_DIR, "processed")
MODELS = os.path.join(ARCHIVE_DIR, "models")
FIGS   = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(MODELS, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

# ----- Load splits -----
d = np.load(os.path.join(PROC, "splits_flatten_32x32x3.npz"))
Xtr, ytr = d["Xtr"], d["ytr"]
Xval, yval = d["Xval"], d["yval"]

# -----  class names for nicer plots -----
label_map_p = os.path.join(PROC, "label_map.json")
idx_to_name = None
if os.path.exists(label_map_p):
    with open(label_map_p, "r") as f:
        idx_to_name = json.load(f)

def _names(labels):
    if idx_to_name is None: return labels
    return [idx_to_name.get(str(i), str(i)) for i in labels]

# ----- Make a fair subset of TRAIN (e.g., 25%) -----
subset_frac = 0.25
sss = StratifiedShuffleSplit(n_splits=1, train_size=subset_frac, random_state=42)
(tr_idx, _), = sss.split(Xtr, ytr)
Xs, ys = Xtr[tr_idx], ytr[tr_idx]
print(f"[Subset] Using {len(ys)} samples ({subset_frac*100:.0f}% of TRAIN) for fitting.")

# ----- Two pipelines: MinMax vs Standard -----
common_mlp = dict(hidden_layer_sizes=(100,), activation="relu", solver="adam",
                  max_iter=200, early_stopping=True, validation_fraction=0.1,
                  random_state=42)

pipelines = {
    "mlp_minmax": Pipeline([
        ("scale", MinMaxScaler()),          # [0,1]
        ("mlp",   MLPClassifier(**common_mlp))
    ]),
    "mlp_std": Pipeline([
        ("scale", StandardScaler()),        # z-score
        ("mlp",   MLPClassifier(**common_mlp))
    ])
}

def fit_eval(tag, pipe, X_fit, y_fit, X_eval, y_eval, save_conf=False):
    t0 = time.time()
    pipe.fit(X_fit, y_fit)
    fit_time = time.time() - t0

    yhat = pipe.predict(X_eval)
    acc  = accuracy_score(y_eval, yhat)
    f1m  = f1_score(y_eval, yhat, average="macro")
    pm   = precision_score(y_eval, yhat, average="macro", zero_division=0)
    rm   = recall_score(y_eval, yhat, average="macro", zero_division=0)
    niter = pipe.named_steps["mlp"].n_iter_

    print(f"[{tag}] Val: acc={acc:.4f}  F1={f1m:.4f}  P={pm:.4f}  R={rm:.4f}  time={fit_time:.1f}s  iters={niter}")

    # save VAL confusion matrices
    if save_conf:
        labels = np.unique(y_eval)
        names  = _names(labels)
        for norm in [None, "true"]:
            cm = confusion_matrix(y_eval, yhat, labels=labels, normalize=norm)
            disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=names)
            fig, ax = plt.subplots(figsize=(8,6))
            disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=90)
            ax.set_title(f"{tag} VAL Confusion Matrix (normalize={norm})")
            plt.tight_layout()
            outpng = os.path.join(FIGS, f"{tag}_val_confmat_{'norm' if norm else 'raw'}.png")
            plt.savefig(outpng, dpi=150); plt.close(fig)

    return {
        "model": tag,
        "split": "Val",
        "n_train_subset": int(len(y_fit)),
        "accuracy": float(acc),
        "macro_f1": float(f1m),
        "macro_precision": float(pm),
        "macro_recall": float(rm),
        "fit_time_sec": float(fit_time),
        "n_iter": int(niter),
    }, pipe

# ----- Run both, export summary CSV -----
results = []
models_out = {}
for tag, pipe in pipelines.items():
    res, fitted = fit_eval(tag, pipe, Xs, ys, Xval, yval, save_conf=False)
    results.append(res)
    with open(os.path.join(MODELS, f"{tag}_subset.pkl"), "wb") as f:
        pickle.dump(fitted, f)

import csv
out_csv = os.path.join(MODELS, "mlp_scaling_compare.csv")
with open(out_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    w.writeheader()
    for r in results: w.writerow(r)
print(f"[Saved] {out_csv}")

with open(os.path.join(MODELS, "mlp_scaling_compare.json"), "w") as f:
    json.dump(results, f, indent=2)


[Subset] Using 13179 samples (25% of TRAIN) for fitting.
[mlp_minmax] Val: acc=0.9025  F1=0.8938  P=0.9102  R=0.8812  time=99.3s  iters=79
[mlp_std] Val: acc=0.9140  F1=0.9033  P=0.9193  R=0.8940  time=58.7s  iters=44
[Saved] /content/gdrive/MyDrive/CS306_2025/database/archive/models/mlp_scaling_compare.csv


In [ ]:
# ===========================================
# Make MLP reports
# ===========================================
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Resolve archive directory
ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")

PROC   = os.path.join(ARCHIVE_DIR, "processed")
MODELS = os.path.join(ARCHIVE_DIR, "models")
FIGS   = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(FIGS, exist_ok=True)

# Paths
splits_p = os.path.join(PROC, "splits_flatten_32x32x3.npz")
model_p  = os.path.join(MODELS, "mlp.pkl")
lmap_p   = os.path.join(PROC, "label_map.json")

# --- Load data/model/labels ---
d = np.load(splits_p)
Xtr, ytr = d["X_train"], d["y_train"]
Xva, yva = d["X_val"],   d["y_val"]
Xte, yte = d["X_test"],  d["y_test"]

with open(model_p, "rb") as f:
    mlp = pickle.load(f)

with open(lmap_p, "r", encoding="utf-8") as f:
    idx2name = json.load(f)["idx_to_name"]
target_names = [idx2name[str(i)] for i in range(len(idx2name))]

# --- Predict ---
y_hat_val = mlp.predict(Xva)
y_hat_te  = mlp.predict(Xte)

# --- Reports to CSV ---
rep_val  = classification_report(yva, y_hat_val, target_names=target_names, digits=3, output_dict=True)
rep_test = classification_report(yte, y_hat_te,  target_names=target_names, digits=3, output_dict=True)
pd.DataFrame(rep_val).T.to_csv(os.path.join(FIGS, "mlp_report_val.csv"))
pd.DataFrame(rep_test).T.to_csv(os.path.join(FIGS, "mlp_report_test.csv"))

# --- Confusion matrix (raw counts) ---
cm = confusion_matrix(yte, y_hat_te)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="Blues", cbar=True)
plt.title("Confusion Matrix — MLP (TEST)")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(FIGS, "mlp_confmat_test.png"), dpi=200)
plt.close()

# Normalized confusion matrix as well
cmn = confusion_matrix(yte, y_hat_te, normalize='true')
plt.figure(figsize=(10, 8))
sns.heatmap(cmn, cmap="Blues", cbar=True, vmin=0.0, vmax=1.0)
plt.title("Confusion Matrix — MLP (TEST, normalized)")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(FIGS, "mlp_confmat_test_norm.png"), dpi=200)
plt.close()

# --- metrics.json summary ---
metrics = {
    "model": "MLPClassifier",
    "params": mlp.get_params(),
    "val": {
        "accuracy": float(rep_val["accuracy"]),
        "macro_f1": float(rep_val["macro avg"]["f1-score"]),
        "weighted_f1": float(rep_val["weighted avg"]["f1-score"]),
    },
    "test": {
        "accuracy": float(rep_test["accuracy"]),
        "macro_f1": float(rep_test["macro avg"]["f1-score"]),
        "weighted_f1": float(rep_test["weighted avg"]["f1-score"]),
    }
}
with open(os.path.join(FIGS, "mlp_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("[Saved] mlp_report_(val|test).csv, mlp_confmat_test(.png|_norm.png), mlp_metrics.json")

# --- use fitted model's loss history if present ---
if hasattr(mlp, "loss_curve_") and isinstance(mlp.loss_curve_, (list, tuple)) and len(mlp.loss_curve_) > 0:
    plt.figure()
    plt.plot(mlp.loss_curve_)
    plt.xlabel("Iteration"); plt.ylabel("Training Loss")
    plt.title("MLP Loss Curve (from fitted model)")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGS, "mlp_loss_curve.png"), dpi=200)
    plt.close()
    print("[Saved] mlp_loss_curve.png")
else:
    print("[Info] No loss_curve_ found on fitted MLP; skipping curve.")


[Saved] mlp_report_(val|test).csv, mlp_confmat_test(.png|_norm.png), mlp_metrics.json
[Saved] mlp_loss_curve.png


In [ ]:
# --- F1 Score ---
import os, json, pickle
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import classification_report

def save_perclass_f1(y_true, y_pred, class_names, out_png, title):
    rep = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    df = pd.DataFrame(rep).T.iloc[:-3]
    plt.figure(figsize=(max(10, len(df)*0.25), 4))
    df["f1-score"].plot(kind="bar")
    plt.ylim(0, 1.0); plt.ylabel("F1-score"); plt.title(title)
    plt.tight_layout(); plt.savefig(out_png, dpi=200); plt.close()

# resolve path directory
try:
    _y_true, _y_pred = yte, y_hat_te
    _FIGS = FIGS
    _PROC = PROC
except NameError:
    ARCHIVE_DIR = next((p for p in [
        "/content/drive/MyDrive/CS306_2025/database/archive",
        "/content/gdrive/MyDrive/CS306_2025/database/archive",
    ] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")
    _PROC  = os.path.join(ARCHIVE_DIR, "processed")
    _FIGS  = os.path.join(ARCHIVE_DIR, "report", "figures")
    os.makedirs(_FIGS, exist_ok=True)

    d = np.load(os.path.join(_PROC, "splits_flatten_32x32x3.npz"))
    Xte, _y_true = d["X_test"], d["y_test"]
    with open(os.path.join(ARCHIVE_DIR, "models", "mlp.pkl"), "rb") as f: mlp = pickle.load(f)
    _y_pred = mlp.predict(Xte)

with open(os.path.join(_PROC, "label_map.json"), "r", encoding="utf-8") as f:
    idx2name = json.load(f)["idx_to_name"]
class_names = [idx2name[str(i)] for i in range(len(idx2name))]

save_perclass_f1(_y_true, _y_pred, class_names,
                 os.path.join(_FIGS, "mlp_f1_per_class.png"),
                 "MLP — per-class F1 (TEST)")
print("[Saved] mlp_f1_per_class.png")


[Saved] mlp_f1_per_class.png


Comparing the models

In [ ]:
# Quick SVM kernel comparison (we did on step 3 to choose the best one, but to save the values to compare)

import os, pickle, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
import json

ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")

PROC   = os.path.join(ARCHIVE_DIR, "processed")
MODELS = os.path.join(ARCHIVE_DIR, "models")
FIGS   = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(MODELS, exist_ok=True); os.makedirs(FIGS, exist_ok=True)

# 1) load splits
d = np.load(os.path.join(PROC, "splits_flatten_32x32x3.npz"))
Xtr, ytr = d["X_train"], d["y_train"]
Xva, yva = d["X_val"],   d["y_val"]

# 2) get standardized inputs for SVM
std_cache = os.path.join(PROC, "svm_arrays_standardized.npz")
scaler_p  = os.path.join(PROC, "svm_scaler.pkl")
if os.path.exists(std_cache):
    dd = np.load(std_cache)
    Xtr_s, Xva_s = dd["X_train_std"], dd["X_val_std"]
else:
    with open(scaler_p, "rb") as f:
        scaler = pickle.load(f)
    Xtr_s, Xva_s = scaler.transform(Xtr), scaler.transform(Xva)

# 3) subset for quick selection (25%)
idx = np.random.RandomState(42).choice(len(Xtr_s), size=int(len(Xtr_s)*0.25), replace=False)
Xsel, ysel = Xtr_s[idx], ytr[idx]

configs = [
    ("linear_sel", SVC(kernel="linear", C=1, random_state=42)),
    ("poly_sel",   SVC(kernel="poly",   C=1, degree=3, gamma="scale", random_state=42)),
    ("rbf_sel",    SVC(kernel="rbf",    C=1, gamma="scale", random_state=42)),
]

for name, clf in configs:
    print(f"\n[SEL] Train SVM-{name} on 25% subset …")
    clf.fit(Xsel, ysel)
    yv = clf.predict(Xva_s)
    acc = accuracy_score(yva, yv)
    print(f"[VAL] {name} acc = {acc*100:.2f}%")
    print(classification_report(yva, yv, digits=4))
    # save light models for plotting
    with open(os.path.join(MODELS, f"svm_{name}.pkl"), "wb") as f:
        pickle.dump(clf, f)

sel_rows = []
for name, clf in configs:
    pth = os.path.join(MODELS, f"svm_{name}.pkl")
    if os.path.exists(pth):
        with open(pth, "rb") as f:
            m = pickle.load(f)
        yv = m.predict(Xva_s)
        acc = accuracy_score(yva, yv)
        sel_rows.append({"name": name, "val_acc": float(acc)})

out_json = os.path.join(MODELS, "svm_selection_metrics.json")
with open(out_json, "w") as f:
    json.dump(sel_rows, f, indent=2)
print(f"[Saved] {out_json}")



[SEL] Train SVM-linear_sel on 25% subset …
[VAL] linear_sel acc = 91.31%
              precision    recall  f1-score   support

           0     0.8049    0.9167    0.8571        36
           1     0.8508    0.9288    0.8881       393
           2     0.8080    0.8660    0.8360       209
           3     0.8767    0.8972    0.8868       214
           4     0.9349    0.8980    0.9161       304
           5     0.8384    0.8861    0.8616       281
           6     0.9231    0.9677    0.9449        62
           7     0.9276    0.9361    0.9318       219
           8     0.9400    0.8785    0.9082       214
           9     0.9417    0.9417    0.9417       223
          10     0.9792    0.9246    0.9511       305
          11     0.9303    0.9350    0.9327       200
          12     0.9519    0.9310    0.9414       319
          13     0.9721    0.9632    0.9676       326
          14     0.9032    0.9492    0.9256       118
          15     0.8696    0.8511    0.8602        94
       

 Comparison

In [ ]:
# ===========================================
# Step 5: Compare & Plot
# - Load saved models (SVMs, MLP)
# - Evaluate on TEST (and use proper preprocessing per model)
# - Save confusion matrices and an accuracy bar chart
# ===========================================


import os, sys, pickle, json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix
# (StandardScaler only needed when transforming at eval time)
from sklearn.preprocessing import StandardScaler

# ---------- Resolve ARCHIVE_DIR automatically ----------
ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")
print("[Info] Using ARCHIVE_DIR:", ARCHIVE_DIR)

# ---------- Paths ----------
PROCESSED_DIR  = os.path.join(ARCHIVE_DIR, "processed")
MODELS_DIR     = os.path.join(ARCHIVE_DIR, "models")
REPORT_DIR     = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(REPORT_DIR, exist_ok=True)

SPLITS_PATH        = os.path.join(PROCESSED_DIR, "splits_flatten_32x32x3.npz")
SVM_STD_CACHE_PATH = os.path.join(PROCESSED_DIR, "svm_arrays_standardized.npz")
SVM_SCALER_PATH    = os.path.join(PROCESSED_DIR, "svm_scaler.pkl")
MLP_SCALER_PATH    = os.path.join(PROCESSED_DIR, "mlp_scaler.pkl")  # present if standardized MLP retrain

SEL_JSON_PATH      = os.path.join(MODELS_DIR, "svm_selection_metrics.json")
SEL_PKLS           = {
    "svm_linear_sel": os.path.join(MODELS_DIR, "svm_linear_sel.pkl"),
    "svm_poly_sel":   os.path.join(MODELS_DIR, "svm_poly_sel.pkl"),
    "svm_rbf_sel":    os.path.join(MODELS_DIR, "svm_rbf_sel.pkl"),
}

# Final models (fair, full-train vs full-train)
SVM_MODELS = {
    "svm_final": os.path.join(MODELS_DIR, "svm_final.pkl"),
}
MLP_MODELS = {
    # prefer standardized MLP if exists; fallback to raw MLP
    "mlp_std":  os.path.join(MODELS_DIR, "mlp_std.pkl"),
    "mlp":      os.path.join(MODELS_DIR, "mlp.pkl"),
}

# ---------- Utils ----------
def pct(x):
    return f"{x*100:.2f}%"

def load_splits():
    if not os.path.exists(SPLITS_PATH):
        print(f"[Error] Missing splits: {SPLITS_PATH}", file=sys.stderr)
        sys.exit(1)
    d = np.load(SPLITS_PATH)
    return d["X_train"], d["y_train"], d["X_val"], d["y_val"], d["X_test"], d["y_test"]

def ensure_svm_std_arrays(X_train, X_val, X_test):
    """Return standardized arrays for SVM evaluation."""
    if os.path.exists(SVM_STD_CACHE_PATH):
        d = np.load(SVM_STD_CACHE_PATH)
        return d["X_train_std"], d["X_val_std"], d["X_test_std"]
    if os.path.exists(SVM_SCALER_PATH):
        with open(SVM_SCALER_PATH, "rb") as f:
            scaler = pickle.load(f)
        return scaler.transform(X_train), scaler.transform(X_val), scaler.transform(X_test)
    print("[Warn] SVM scaler/cache not found; evaluating SVM on raw inputs (may reduce accuracy).")
    return X_train, X_val, X_test

def maybe_standardize_for_mlp(X_train, X_val, X_test):
    """If standardized-MLP exists, transform inputs; else return raw arrays."""
    if os.path.exists(MLP_SCALER_PATH):
        with open(MLP_SCALER_PATH, "rb") as f:
            s = pickle.load(f)
        return s.transform(X_train), s.transform(X_val), s.transform(X_test), True
    return X_train, X_val, X_test, False

def plot_confmat(cm, title, out_png):
    plt.figure(figsize=(6,5))
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()
    print(f"[Saved] {out_png}")

# ---------- Selection (VAL) chart ----------
def selection_val_chart(X_val_std, y_val):
    """
    Make a validation bar chart for SVM kernel selection.
    Priority: use JSON metrics if present; otherwise load *_sel.pkl and compute.
    """
    sel_results = []

    if os.path.exists(SEL_JSON_PATH):
        try:
            with open(SEL_JSON_PATH, "r") as f:
                js = json.load(f)
            # Expect [{"name": "linear_sel", "val_acc": 0.976, ...}, ...]
            sel_results = [(d["name"], float(d["val_acc"])) for d in js if "val_acc" in d]
            print("[Info] Loaded selection metrics from JSON.")
        except Exception as e:
            print("[Warn] Failed to read selection JSON, will try *_sel.pkl:", e)

    if not sel_results:
        # Fallback: compute from saved *_sel.pkl models
        for name, pth in SEL_PKLS.items():
            if not os.path.exists(pth):
                print(f"[Info] skip (no selection model): {pth}")
                continue
            with open(pth, "rb") as f:
                clf = pickle.load(f)
            yv = clf.predict(X_val_std)
            acc = accuracy_score(y_val, yv)
            sel_results.append((name, acc))
        if sel_results:
            print("[Info] Computed selection metrics from *_sel.pkl.")

    if not sel_results:
        print("[Info] No selection metrics/models found; skip VAL chart.")
        return

    # Sort and plot
    labels = [k for k,_ in sel_results]
    accs   = [v for _,v in sel_results]
    order  = np.argsort(accs)[::-1]
    labels = [labels[i] for i in order]
    accs   = [accs[i] for i in order]

    plt.figure(figsize=(6,4))
    plt.bar(labels, accs)
    plt.ylabel("Validation Accuracy")
    plt.ylim(0.0, 1.0)
    plt.title("SVM kernel comparison (25% subset, VAL)")
    plt.xticks(rotation=20)
    plt.tight_layout()
    out_png = os.path.join(REPORT_DIR, "svm_sel_val_bar.png")
    plt.savefig(out_png, dpi=150)
    plt.close()
    print(f"[Saved] {out_png}")

def plot_confmat_norm(y_true, y_pred, title, out_png):
    cmn = confusion_matrix(y_true, y_pred, normalize='true')  # row-normalized
    plt.figure(figsize=(6,5))
    plt.imshow(cmn, interpolation='nearest')
    plt.title(title + " (normalized)")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)  # consider dpi=300 if labels are dense
    plt.close()


# ---------- Main ----------
def main():
    print("5): Compare & Plot")
    X_train, y_train, X_val, y_val, X_test, y_test = load_splits()

    # Prepare inputs
    Xtr_svm, Xva_svm, Xte_svm = ensure_svm_std_arrays(X_train, X_val, X_test)
    Xtr_mlp, Xva_mlp, Xte_mlp, mlp_is_std = maybe_standardize_for_mlp(X_train, X_val, X_test)

    # ----- SELECTION (VAL) CHART -----
    selection_val_chart(Xva_svm, y_val)  # uses JSON or *_sel.pkl if present

    # -----  svm_final vs mlp -----
    models = {}

    # Load SVM final (Linear)
    svm_final_path = SVM_MODELS["svm_final"]
    if os.path.exists(svm_final_path):
        with open(svm_final_path, "rb") as f:
            models["svm_final"] = pickle.load(f)
    else:
        print(f"[Error] Missing final SVM: {svm_final_path}", file=sys.stderr)

    # Load MLP
    if os.path.exists(MLP_MODELS["mlp_std"]):
        with open(MLP_MODELS["mlp_std"], "rb") as f:
            models["mlp_std"] = pickle.load(f)
        mlp_key = "mlp_std"
    elif os.path.exists(MLP_MODELS["mlp"]):
        with open(MLP_MODELS["mlp"], "rb") as f:
            models["mlp"] = pickle.load(f)
        mlp_key = "mlp"
    else:
        mlp_key = None
        print(f"[Error] Missing MLP model (mlp_std.pkl or mlp.pkl).", file=sys.stderr)

    if not models or "svm_final" not in models or mlp_key is None:
        print("[Error] Required final models not found; abort.", file=sys.stderr)
        sys.exit(1)

    # Evaluation
    results = []
    for name, clf in models.items():
        Xte = Xte_svm if name.startswith("svm") else Xte_mlp
        yp = clf.predict(Xte)
        acc = accuracy_score(y_test, yp)
        results.append((name, acc))

        cm = confusion_matrix(y_test, yp)
        plot_confmat(cm, f"{name} Confusion Matrix (TEST)",
                     os.path.join(REPORT_DIR, f"confmat_{name}.png"))
        plot_confmat_norm(y_test, yp,
                          f"{name} Confusion Matrix (TEST)",
                          os.path.join(REPORT_DIR, f"confmat_{name}_norm.png"))

        print(f"[Test] {name:10s} acc = {pct(acc)} (cm shape={cm.shape})")

    # Accuracy bar graph
    labels = [r[0] for r in results]
    accs   = [r[1] for r in results]
    order  = np.argsort(accs)[::-1]
    labels = [labels[i] for i in order]
    accs   = [accs[i] for i in order]

    plt.figure(figsize=(6,4))
    plt.bar(labels, accs)
    plt.ylabel("Test Accuracy")
    plt.ylim(0.0, 1.0)
    plt.title("Final models (FULL) — Test accuracy")
    plt.xticks(rotation=20)
    plt.tight_layout()
    out_png = os.path.join(REPORT_DIR, "accuracy_bar_FINAL.png")
    plt.savefig(out_png, dpi=150)
    plt.close()
    print(f"[Saved] {out_png}")

    # Summary
    print("\n[Summary] Test accuracy (final models)")
    for name, acc in results:
        print(f"  - {name:10s}: {pct(acc)}")

if __name__ == "__main__":
    main()








[Info] Using ARCHIVE_DIR: /content/drive/MyDrive/CS306_2025/database/archive
5): Compare & Plot
[Info] Loaded selection metrics from JSON.
[Saved] /content/drive/MyDrive/CS306_2025/database/archive/report/figures/svm_sel_val_bar.png
[Saved] /content/drive/MyDrive/CS306_2025/database/archive/report/figures/confmat_svm_final.png
[Test] svm_final  acc = 97.60% (cm shape=(43, 43))
[Saved] /content/drive/MyDrive/CS306_2025/database/archive/report/figures/confmat_mlp.png
[Test] mlp        acc = 95.02% (cm shape=(43, 43))
[Saved] /content/drive/MyDrive/CS306_2025/database/archive/report/figures/accuracy_bar_FINAL.png

[Summary] Test accuracy (final models)
  - svm_final : 97.60%
  - mlp       : 95.02%


In [ ]:
# Final comparison CSV (SVM vs MLP)
import os, json, pandas as pd

ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")

REPORT_DIR = os.path.join(ARCHIVE_DIR, "report", "figures")

rows = []

# SVM
svm_json = os.path.join(REPORT_DIR, "svm_final_metrics.json")
if os.path.exists(svm_json):
    with open(svm_json) as f: m = json.load(f)
    rows.append({
        "model": "svm_final",
        "test_accuracy": m["test"]["accuracy"],
        "test_macro_f1": m["test"]["macro_f1"]
    })
# MLP
mlp_json = os.path.join(REPORT_DIR, "mlp_metrics.json")
if os.path.exists(mlp_json):
    with open(mlp_json) as f: m = json.load(f)
    rows.append({
        "model": "mlp",
        "test_accuracy": m["test"]["accuracy"],
        "test_macro_f1": m["test"]["macro_f1"]
    })sv

df = pd.DataFrame(rows)
out_csv = os.path.join(REPORT_DIR, "final_comparison.csv")
df.to_csv(out_csv, index=False)
print(f"[Saved] {out_csv}")
df


[Saved] /content/drive/MyDrive/CS306_2025/database/archive/report/figures/final_comparison.csv


,model,test_accuracy,test_macro_f1
0,svm_final,0.976033,0.975115
1,mlp,0.950154,0.945072


In [ ]:

# FINAL COMPARISON (Linear-SVM final, RBF-SVM(c10), MLP)

import os, json, sys, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split


ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")
print("[Info] Using ARCHIVE_DIR:", ARCHIVE_DIR)

# ---------- Paths ----------
PROC   = os.path.join(ARCHIVE_DIR, "processed")
MODELS = os.path.join(ARCHIVE_DIR, "models")
FIGS   = os.path.join(ARCHIVE_DIR, "report", "figures")
os.makedirs(FIGS, exist_ok=True)

# ---------- Robust split loader ----------
def load_splits(proc_dir, seed=42):
    """
    Try splits_* .npz with flexible key names.
    Fallback: arrays_* .npz with X,y -> build 72/8/20 here.
    Returns: Xtr, ytr, Xval, yval, Xte, yte
    """
    # 1) Try known splits files
    for fname in ["splits_flatten_32x32x3.npz", "splits_32x32_flatten.npz", "splits.npz"]:
        p = os.path.join(proc_dir, fname)
        if not os.path.exists(p):
            continue
        d = np.load(p)
        keys = set(d.files)
        # common variants seen so far
        candidates = [
            ("Xtr","Xval","Xte","ytr","yval","yte"),
            ("X_train","X_val","X_test","y_train","y_val","y_test"),
            ("Xtrain","Xval","Xtest","Ytrain","Yval","Ytest"),
            ("XTr","XVal","XTe","yTr","yVal","yTe"),
        ]
        for XtrK, XvalK, XteK, ytrK, yvalK, yteK in candidates:
            if {XtrK,XvalK,XteK,ytrK,yvalK,yteK}.issubset(keys):
                print(f"[Info] Using splits from: {fname}")
                return d[XtrK], d[ytrK], d[XvalK], d[yvalK], d[XteK], d[yteK]
        print(f"[Warn] Found '{fname}' but expected keys not present. keys={sorted(keys)}")

    # 2) Fallback: arrays_* with X,y
    for fname in ["arrays_flatten_32x32x3.npz", "arrays_32x32_flatten.npz", "arrays.npz"]:
        p = os.path.join(proc_dir, fname)
        if not os.path.exists(p):
            continue
        d = np.load(p)
        keys = set(d.files)
        Xkey = next((k for k in ["X","Xall","X_all","features"] if k in keys), None)
        ykey = next((k for k in ["y","yall","y_all","labels"] if k in keys), None)
        if Xkey and ykey:
            print(f"[Info] Building splits (72/8/20) from arrays: {fname}")
            X, y = d[Xkey], d[ykey]
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)
            X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.10, stratify=y_tr, random_state=seed)
            return X_tr, y_tr, X_val, y_val, X_te, y_te
        else:
            print(f"[Warn] '{fname}' lacks obvious X/y keys. keys={sorted(keys)}")

    raise FileNotFoundError("Could not load splits. Provide splits_* .npz with X*/y* or arrays_* .npz with X,y.")

# ---------- Label map  ----------
def load_label_names(proc_dir):
    # Prefer flat; else nested; else None
    for fname in ["label_map_flat.json", "label_map.json"]:
        p = os.path.join(proc_dir, fname)
        if os.path.exists(p):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    lm = json.load(f)
                if "idx_to_name" in lm:     # nested structure
                    lm = lm["idx_to_name"]
                # ensure str keys
                return {str(k): v for k, v in lm.items()}
            except Exception as e:
                print("[Warn] Failed to read label map:", e)
    return None

# ---------- Metrics/plots helpers ----------
def export_report_and_metrics(model_tag, y_true, y_pred, out_dir):
    """
    Saves:
      - report CSV (per-class + macro/weighted/accuracy)
      - metrics JSON (acc, macro_F1/P/R)
      - confusion matrices (raw & normalized)
    Returns a summary dict for the comparison table.
    """
    acc  = accuracy_score(y_true, y_pred)
    f1m  = f1_score(y_true, y_pred, average="macro")
    pm   = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rm   = recall_score(y_true, y_pred, average="macro", zero_division=0)

    rep = classification_report(y_true, y_pred, digits=4, output_dict=True)
    pd.DataFrame(rep).T.to_csv(os.path.join(out_dir, f"{model_tag}_report_test.csv"))

    with open(os.path.join(out_dir, f"{model_tag}_metrics.json"), "w") as f:
        json.dump({
            "model": model_tag,
            "test": {
                "accuracy": acc,
                "macro_f1": f1m,
                "macro_precision": pm,
                "macro_recall": rm
            }
        }, f, indent=2)

    # Confusion matrices (raw & normalized)
    labels_sorted = np.unique(y_true)
    label_map = load_label_names(PROC)
    display_names = [label_map.get(str(i), str(i)) for i in labels_sorted] if label_map else labels_sorted

    for norm_tag, suffix in [(None, "raw"), ("true", "norm")]:
        cm = confusion_matrix(y_true, y_pred, labels=labels_sorted, normalize=norm_tag)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_names)
        fig, ax = plt.subplots(figsize=(8,6))
        disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=90)
        ax.set_title(f"{model_tag} — Confusion Matrix (normalize={norm_tag})")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"confmat_{model_tag}_{suffix}.png"), dpi=160)
        plt.close(fig)

    return {
        "model": model_tag,
        "test_accuracy": acc,
        "test_macro_f1": f1m,
        "test_macro_precision": pm,
        "test_macro_recall": rm
    }

# ---------- Input routing ----------
def needs_raw_input(clf):
    """
    If pipeline already has a scaler/PCA inside, prefer feeding RAW arrays once.
    """
    try:
        steps = getattr(clf, "named_steps", {})
        return bool(steps)
    except Exception:
        return False  # not a pipeline

# ---------- MAIN ----------
def main():
    # 1) Load splits
    Xtr, ytr, Xval, yval, Xte, yte = load_splits(PROC)
    print("[Shapes]", { "Xtr": Xtr.shape, "Xval": Xval.shape, "Xte": Xte.shape })


    model_paths = {
        "svm_linear_final": os.path.join(MODELS, "svm_final.pkl"),
        "svm_rbf_c10":      os.path.join(MODELS, "svm_rbf_c10.pkl"),
        "mlp_std":          os.path.join(MODELS, "mlp_std.pkl"),
        "mlp":              os.path.join(MODELS, "mlp.pkl"),
    }
    models = {}
    for tag, p in model_paths.items():
        if os.path.exists(p):
            with open(p, "rb") as f:
                models[tag] = pickle.load(f)
        else:
            print(f"[Info] Missing (skip): {p}")

    if not models:
        print("[Error] No models found to compare.", file=sys.stderr); return

    # 3) Evaluate on the SAME TEST SET & export artifacts
    rows = []
    for tag, clf in models.items():
        X_in = Xte if needs_raw_input(clf) else Xte
        t0 = time.time()
        yhat = clf.predict(X_in)
        infer_time = time.time() - t0

        summary = export_report_and_metrics(tag, yte, yhat, FIGS)
        summary["infer_time_sec_on_test"] = infer_time
        rows.append(summary)
        print(f"[Test] {tag:16s} acc={summary['test_accuracy']:.4f}  "
              f"F1={summary['test_macro_f1']:.4f}  P={summary['test_macro_precision']:.4f}  "
              f"R={summary['test_macro_recall']:.4f}  (pred_time={infer_time:.2f}s)")


    comp_csv = os.path.join(FIGS, "final_comparison_metrics.csv")
    pd.DataFrame(rows).to_csv(comp_csv, index=False)
    print(f"[Saved] {comp_csv}")


    labels = [r["model"] for r in rows]
    accs   = [r["test_accuracy"] for r in rows]
    order  = np.argsort(accs)[::-1]
    labels = [labels[i] for i in order]; accs = [accs[i] for i in order]
    plt.figure(figsize=(6,4))
    plt.bar(labels, accs)
    plt.ylabel("Test Accuracy"); plt.ylim(0.0, 1.0)
    plt.title("Final models — Test accuracy")
    plt.xticks(rotation=20); plt.tight_layout()
    out_png = os.path.join(FIGS, "accuracy_bar_FINAL.png")
    plt.savefig(out_png, dpi=160); plt.close()
    print(f"[Saved] {out_png}")

main()


[Info] Using ARCHIVE_DIR: /content/gdrive/MyDrive/CS306_2025/database/archive
[Info] Using splits from: splits_flatten_32x32x3.npz
[Shapes] {'Xtr': (52718, 3072), 'Xval': (5858, 3072), 'Xte': (14645, 3072)}
[Info] Missing (skip): /content/gdrive/MyDrive/CS306_2025/database/archive/models/mlp_std.pkl


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[Test] svm_linear_final acc=0.2032  F1=0.1145  P=0.3224  R=0.1199  (pred_time=760.29s)
[Test] svm_rbf_c10      acc=0.9849  F1=0.9846  P=0.9893  R=0.9804  (pred_time=1969.21s)
[Test] mlp              acc=0.9502  F1=0.9451  P=0.9530  R=0.9414  (pred_time=25.97s)
[Saved] /content/gdrive/MyDrive/CS306_2025/database/archive/report/figures/final_comparison_metrics.csv
[Saved] /content/gdrive/MyDrive/CS306_2025/database/archive/report/figures/accuracy_bar_FINAL.png


In [ ]:
#  Log environment versions
import os, json, numpy, sklearn, sys, platform

ARCHIVE_DIR = next((p for p in [
    "/content/drive/MyDrive/CS306_2025/database/archive",
    "/content/gdrive/MyDrive/CS306_2025/database/archive",
] if os.path.exists(p)), "/content/drive/MyDrive/CS306_2025/database/archive")
REPORT_DIR = os.path.join(ARCHIVE_DIR, "report", "figures")

try:
    import tensorflow as tf
    tfv = tf.__version__
except Exception:
    tfv = "n/a"

env = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": numpy.__version__,
    "sklearn": sklearn.__version__,
    "tensorflow": tfv
}

out_json = os.path.join(REPORT_DIR, "env_versions.json")
with open(out_json, "w") as f:
    json.dump(env, f, indent=2)
print(f"[Saved] {out_json}")
env


[Saved] /content/gdrive/MyDrive/CS306_2025/database/archive/report/figures/env_versions.json


{'python': '3.12.11',
 'platform': 'Linux-6.6.97+-x86_64-with-glibc2.35',
 'numpy': '2.0.2',
 'sklearn': '1.6.1',
 'tensorflow': '2.19.0'}